# AI-Assisted Data Analysis

This notebook demonstrates the `ai_client` package:

1. Load a synthetic dataset
2. Ask the AI client a question about it
3. Display the response (blocking and streaming)

**Prerequisites**: The devcontainer must have been built with
`ANTHROPIC_AUTH_TOKEN` set in the local environment.

In [ ]:
import textwrap

import pandas as pd

from ai_client import AIClient, ChatMessage, ChatRequest, ProxyConfig

## 1. Build a synthetic dataset

No external downloads — all data is generated in-memory.

In [ ]:
data = {
    "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun"],
    "revenue_usd": [12_000, 15_400, 11_800, 18_200, 22_500, 19_700],
    "units_sold": [240, 308, 236, 364, 450, 394],
    "returns": [12, 18, 10, 22, 30, 25],
}

df = pd.DataFrame(data)
df["return_rate_pct"] = (df["returns"] / df["units_sold"] * 100).round(2)
df

## 2. Configure the AI client

In [ ]:
config = ProxyConfig.from_env()
client = AIClient(config)
print(f"Connected to: {config.base_url_str}")

## 3. Build the request

In [ ]:
csv_snapshot = df.to_csv(index=False)

question = (
    "Based on the sales data below, identify the month with the highest "
    "return rate and explain one possible business reason for it.\n\n"
    f"```csv\n{csv_snapshot}```"
)

request = ChatRequest(
    model="claude-sonnet-latest",
    messages=[ChatMessage(role="user", content=question)],
    max_tokens=512,
    system="You are a concise data analyst. Answer in at most three sentences.",
)

## 4. Blocking response

In [ ]:
response = client.chat(request)

print(f"Model : {response.model}")
print(f"Tokens: {response.input_tokens} in / {response.output_tokens} out")
print(f"Stop  : {response.stop_reason}")
print()
for block in response.content:
    if block.type == "text":
        print(textwrap.fill(block.text, width=80))

## 5. Streaming response

Same request; tokens are printed as they arrive.

In [ ]:
import sys

print("Streaming response:")
print("-" * 40)
for chunk in client.stream_chat(request):
    print(chunk, end="", flush=True)
print()
print("-" * 40)